In [1]:
# conda activate genomic_tools

import os
import json
import pickle
import pandas as pd
from collections import defaultdict

pd.set_option('display.max_columns', None)

## Load interproscan results

In [2]:
cols = [
    'protein_accession',
    'sequence_md5',
    'sequence_length',
    'analysis',
    'signature_accession',
    'signature_description',
    'start',
    'stop',
    'score',
    'status',
    'date',
    'interpro_accession',
    'interpro_description',
    'go_annotations',
    'pathways'
]

interproscan_results = pd.read_csv(
    "data/interproscan/results/proteins.fa.tsv",
    sep='\t',
    header=None,
    names=cols,
    index_col=False
)

In [3]:
# Parse the PIRSR data

with open("data/interproscan/interpro/data/pirsr/sr_uru.json") as f:
    pirsr_data = json.load(f)

# subset to proteins patterns applicable to humans
human_relevant = ['Eukaryota', 'Eukaryota; Metazoa', 'Eukaryota; Vertebrata', 'Eukaryota; Chordata', 'Eukaryota; Mammalia', 'Eukaryota; Eutheria']

records = []
for ac, entry in pirsr_data.items():
    for group_id, sites in entry['Groups'].items():
        for site in sites:
            scope = entry.get('Scope', [])
            tr = entry.get('TR', '')
            if any(s in human_relevant for s in scope):
                records.append({
                    'accession': ac,
                    'scope':  ', '.join(scope),
                    'TR': tr.split("; ")[1],
                    'label': site['label'],
                    'condition': site['condition'],
                    'desc': site['desc'],
                    'group': group_id
                })

pirsr_df = pd.DataFrame(records)

pirsr_df = pirsr_df.groupby("accession").agg(
    scope=('scope', lambda x: ' | '.join(x.unique())),
    TR=('TR', lambda x: ' | '.join(x.unique())),
    label=('label', lambda x: ' | '.join(x.unique())),
    condition=('condition', lambda x: ' | '.join(x.unique())),
    desc=('desc', lambda x: ' | '.join(x.unique())),
    group=('group', lambda x: ' | '.join(x.unique())),
).reset_index()

In [4]:
interproscan_results = interproscan_results.merge(pirsr_df, left_on="signature_accession", right_on="accession", how="left")

interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'signature_description'] = interproscan_results.loc[interproscan_results['analysis'] == "PIRSR", 'label'] 

In [9]:
interproscan_results[interproscan_results['analysis'] == "SFLD"]

,protein_accession,sequence_md5,sequence_length,analysis,signature_accession,signature_description,start,stop,score,status,date,interpro_accession,interpro_description,go_annotations,pathways,accession,scope,TR,label,condition,desc,group
4402,ENST00000368356,D1FD398E425C19F72605225485301565,419,SFLD,SFLDS00005,Isoprenoid Synthase Type I,75,417,0.0,SFLD,08-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4403,ENST00000368356,D1FD398E425C19F72605225485301565,419,SFLD,SFLDG01017,Polyprenyl Transferase Like,75,417,0.0,SFLD,08-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6380,ENSG00000065621_ProteinCoding_2_synthetic_skip,D31F2E05DACB5358EF3109582DE62244,209,SFLD,SFLDS00019,Glutathione Transferase (cytosolic),24,120,5.7E-29,SFLD,08-08-2026,IPR040079,Glutathione transferase family,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6381,ENST00000450629,D31F2E05DACB5358EF3109582DE62244,209,SFLD,SFLDS00019,Glutathione Transferase (cytosolic),24,120,5.7E-29,SFLD,08-08-2026,IPR040079,Glutathione transferase family,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6382,ENSG00000065621_ProteinCoding_2_synthetic_skip,D31F2E05DACB5358EF3109582DE62244,209,SFLD,SFLDG00358,Main (cytGST),24,120,5.7E-29,SFLD,08-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
605040,ENST00000359149,415C35D92CA1FA096582DEDE953F33C6,1173,SFLD,SFLDG00002,C1.7: P-type atpase like,453,844,0.0,SFLD,08-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
605041,ENST00000359149,415C35D92CA1FA096582DEDE953F33C6,1173,SFLD,SFLDF00027,p-type atpase,453,844,0.0,SFLD,08-08-2026,IPR044492,"P-type ATPase, haloacid dehalogenase domain",-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
605042,ENST00000359149,415C35D92CA1FA096582DEDE953F33C6,1173,SFLD,SFLDS00003,Haloacid Dehalogenase,453,844,0.0,SFLD,08-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN
608943,ENSG00000104313_ProteinCoding_1_synthetic_skip,0AD34E40403EE810788F3DDA40656454,562,SFLD,SFLDG01129,"C1.5: HAD, Beta-PGM, Phosphatase Like",322,561,2.5E-38,SFLD,08-08-2026,-,-,-,-,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [57]:
interproscan_results.shape

(628144, 22)

## Map events to interproscan results

(Get the transcript associated with each significant event)

In [58]:
with open("data/gencode.v46.annotation_cds_by_transcript.pkl", "rb") as f:
    cds_by_transcript = pickle.load(f)

In [59]:
with open('data/event_protein_map.pkl', 'rb') as f:
    event_protein_map = pickle.load(f)

In [60]:
def near_exon(exon_start, exon_end, scan_start, scan_end, window=300):
    """Returns boolean: are features are within (window nts) of exon?"""
    return (abs(scan_start - exon_start) < window) | \
        (abs(scan_end - exon_end) < window)
    
def _cds_rows(obj):
    if obj is None:
        return None
    if hasattr(obj, "iterrows"):
        return [{'start': int(r['start']), 'end': int(r['end']), 'frame': int(r['frame'])}
                for _, r in obj.iterrows()]
    return [{'start': int(c['start']), 'end': int(c['end']), 'frame': int(c['frame'])} for c in obj]

def rel_cds_to_genome(cds_obj, strand):
    """
    Build a list of exon segments mapping between genomic coordinates
    and CDS-relative coordinates, in transcript (5'->3') order.

    Each segment is a dict:
        genome_start, genome_end : genomic coordinates (always genome_start <= genome_end)
        rel_start, rel_end       : CDS-relative coordinates (always rel_start <= rel_end)
    """
    cds = _cds_rows(cds_obj)

    # order exons in transcript (5'->3') order
    if strand == '-':
        cds = sorted(cds, key=lambda c: c['start'], reverse=True)
    else:
        cds = sorted(cds, key=lambda c: c['start'])

    segments = []
    cds_start = 0
    for c in cds:
        length = c['end'] - c['start'] + 1
        segments.append({
            'genome_start': c['start'],
            'genome_end': c['end'],
            'rel_start': cds_start,
            'rel_end': cds_start + length - 1,
        })
        cds_start += length

    return segments

def map_aa_to_genome(rel_to_genome, aa_start, aa_end, strand):
    result = []
    # convert AA position to relative CDS position
    # note: subtract 1 to convert from 1-indexed AA to 0-indexed
    cds_start = (aa_start - 1) * 3
    cds_end = (aa_end - 1) * 3 + 2
    for seg in rel_to_genome:
        # seg contains mapping from relative CDS position to genome position
        overlap_start = max(cds_start, seg['rel_start'])
        overlap_end = min(cds_end, seg['rel_end'])
        if overlap_start <= overlap_end:
            if strand == '-':
                # rel increases as genome decreases
                g_start = seg['genome_end'] - (overlap_end - seg['rel_start'])
                g_end = seg['genome_end'] - (overlap_start - seg['rel_start'])
            else:
                # rel increases as genome increases
                g_start = seg['genome_start'] + (overlap_start - seg['rel_start'])
                g_end = seg['genome_start'] + (overlap_end - seg['rel_start'])
            result.append((g_start, g_end))
    return result

In [ ]:
# map interproscan results to their corresponding transcript (representing the splicing event of interest)
columns = ['protein_accession', 'sequence_length', 'analysis', 
           'signature_description', 'start', 'stop', 'interpro_description']
ipr = interproscan_results.loc[:, columns]

ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

event_interproscan_map = defaultdict(dict)

for ev, rec in event_protein_map.items():
    
    rec_incl = rec['inclusion']
    incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
    if incl_df is None:
        continue
    overlap_df = incl_df[
        (incl_df['start'] <= rec_incl['aa_end']) &
        (incl_df['stop']  >= rec_incl['aa_start'])
    ]
    
    
    if not overlap_df.empty:
        strand = rec['meta']['strand']

        # save AA position of protein domains in terms of genomic coordinates
        cds_obj = cds_by_transcript[rec_incl['transcript_id']]
        cds_to_genome = rel_cds_to_genome(cds_obj, strand)
        genome_coords = []
        for idx, row in overlap_df.iterrows():
            aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
            genome_coords.append(aa_to_genome)
    
        incl_cols = ['aa_start', 'aa_end', 'exon_cds_start', 'exon_cds_end',
                    'frame_preserving', 'clean_start', 'clean_end']
        
        event_interproscan_map[ev] = {
            'meta': rec['meta'],
            'inclusion': overlap_df.assign(
                **{col: rec_incl[col] for col in incl_cols},
                genome_coords=genome_coords
            ).reset_index(drop=True),
            'real_skip': None,
            'synthetic_skip': None,
            'exon_diff_junction_siblings': None,
            'exon_diff_boundary_siblings': None
        }
    
        # real skip
        if rec.get('real_skip'):
            skip_df = ipr_grouped.get(rec['real_skip'])
            if skip_df is not None: 
                cds_obj = cds_by_transcript[rec['real_skip']]
                cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                genome_coords = []
                for idx, row in skip_df.iterrows():
                    # there's no AA position of the exon to check for overlaps because it was skipped here!
                    # instead, check if domain hits are in the genomic neighborhood of the original exon
                    # note: if real skip transcript is very different 5' structure, the original exon neighborhood may be out of scope
                    aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                    scan_start = min(min(t) for t in aa_to_genome)
                    scan_end = max(max(t) for t in aa_to_genome)
                    if near_exon(rec_incl['exon_cds_start'], rec_incl['exon_cds_end'], scan_start, scan_end): 
                        genome_coords.append(aa_to_genome)
                    else:
                        genome_coords.append(None)
                        
                skip_df = skip_df.assign(genome_coords=genome_coords)
                skip_df = skip_df[skip_df['genome_coords'].notna()]
                    
                if not skip_df.empty:
                    event_interproscan_map[ev]['real_skip'] = skip_df.reset_index(drop=True)
                    
        # synthetic skip
        if rec.get('synthetic_skip'):
            synth_df = ipr_grouped.get(rec['synthetic_skip'])
            if synth_df is not None:
                # there's no AA position of the exon to check for overlaps because it was skipped here!
                # instead, check if domain hits are in the genomic neighborhood of the original exon
                cds_obj = cds_by_transcript[rec['inclusion']['transcript_id']]
                es, ee = rec['meta']['es'], rec['meta']['ee']
                mask = (cds_obj['start'] == es) & (cds_obj['end'] == ee)
                cds_obj = cds_obj[~mask]
                cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                genome_coords = []
                for idx, row in synth_df.iterrows():
                    aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                    scan_start = min(min(t) for t in aa_to_genome)
                    scan_end = max(max(t) for t in aa_to_genome)
                    if near_exon(rec_incl['exon_cds_start'], rec_incl['exon_cds_end'], scan_start, scan_end): 
                        genome_coords.append(aa_to_genome)
                    else:
                        genome_coords.append(None)
                        
                synth_df = synth_df.assign(genome_coords=genome_coords)
                synth_df = synth_df[synth_df['genome_coords'].notna()]
                
                if not synth_df.empty:
                    event_interproscan_map[ev]['synthetic_skip'] = synth_df.reset_index(drop=True)
        
        # junction siblings
        if rec.get('exon_diff_junction_siblings'):
            sib_frames = []
            for sib in rec['exon_diff_junction_siblings']:
                t = sib['transcript_id']
                sib_df = ipr_grouped.get(t) 
                if sib_df is None:
                    continue
                sib_overlap = near_exon(sib_df, sib['aa_start'], sib['aa_end'], window=50)
                # sib_overlap = sib_df[
                #     (sib_df['start'] <= sib['aa_end']) &
                #     (sib_df['stop']  >= sib['aa_start'])
                # ]
                if not sib_overlap.empty:
                    # save AA position of protein domains in terms of genomic coordinates
                    cds_obj = cds_by_transcript[t]
                    cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                    genome_coords = []
                    for idx, row in sib_overlap.iterrows():
                        aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                        genome_coords.append(aa_to_genome)

                    sib_frames.append(sib_overlap.assign(
                        aa_start=sib['aa_start'],
                        aa_end=sib['aa_end'],
                        exon_cds_start=sib['exon_cds_start'],
                        exon_cds_end=sib['exon_cds_end'],
                        genome_coords=genome_coords
                    ).reset_index(drop=True))

            if sib_frames:
                event_interproscan_map[ev]['exon_diff_junction_siblings'] = pd.concat(sib_frames, ignore_index=True)

        # boundary siblings
        if rec.get('exon_diff_boundary_siblings'):
            sib_frames = []
            for sib in rec['exon_diff_boundary_siblings']:
                t = sib['transcript_id']
                sib_df = ipr_grouped.get(t) 
                if sib_df is None:
                    continue
                sib_overlap = near_exon(sib_df, sib['aa_start'], sib['aa_end'], window=50)
                # sib_overlap = sib_df[
                #     (sib_df['start'] <= sib['aa_end']) &
                #     (sib_df['stop']  >= sib['aa_start'])
                # ]
                if not sib_overlap.empty:
                    # save AA position of protein domains in terms of genomic coordinates
                    cds_obj = cds_by_transcript[t]
                    cds_to_genome = rel_cds_to_genome(cds_obj, strand)
                    genome_coords = []
                    for idx, row in sib_overlap.iterrows():
                        aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
                        genome_coords.append(aa_to_genome)

                    sib_frames.append(sib_overlap.assign(
                        aa_start=sib['aa_start'],
                        aa_end=sib['aa_end'],
                        exon_cds_start=sib['exon_cds_start'],
                        exon_cds_end=sib['exon_cds_end'],
                        genome_coords=genome_coords
                    ).reset_index(drop=True))

            if sib_frames:
                event_interproscan_map[ev]['exon_diff_boundary_siblings'] = pd.concat(sib_frames, ignore_index=True)

Debug

In [ ]:
# ev = "ENSG00000285043_ProteinCoding_1"
# rec = event_protein_map[ev]

In [ ]:
# # map interproscan results to their corresponding transcript (representing the splicing event of interest)
# analyses_to_exclude = ['NCBIFAM', 'SFLD']
# columns = ['protein_accession', 'sequence_length', 'analysis', 
#            'signature_description', 'start', 'stop', 'interpro_description']
# ipr = interproscan_results.loc[~interproscan_results['analysis'].isin(analyses_to_exclude), columns]

# ipr_grouped = {acc: grp for acc, grp in ipr.groupby('protein_accession')}

# event_interproscan_map = defaultdict(dict)

# rec_incl = rec['inclusion']
# incl_df  = ipr_grouped.get(rec_incl['transcript_id'])
# if incl_df is None:
#     continue
# overlap_df = incl_df[
#     (incl_df['start'] <= rec_incl['aa_end']) &
#     (incl_df['stop']  >= rec_incl['aa_start'])
# ]



# strand = rec['meta']['strand']

In [ ]:
# # junction siblings

# for sib in rec['exon_diff_junction_siblings']:
#     t = sib['transcript_id']
#     sib_df = ipr_grouped.get(t) 
#     if sib_df is None:
#         continue
#     sib_overlap = sib_df[
#         (sib_df['start'] <= sib['aa_end']) &
#         (sib_df['stop']  >= sib['aa_start'])
#     ]
#     break


In [ ]:
# # save AA position of protein domains in terms of genomic coordinates
# cds_obj = cds_by_transcript[t]
# cds_to_genome = rel_cds_to_genome(cds_obj, strand)
# genome_coords = []
# for idx, row in sib_overlap.iterrows():
#     aa_to_genome = map_aa_to_genome(cds_to_genome, row['start'], row['stop'], strand)
#     genome_coords.append(aa_to_genome)

In [47]:
len(event_interproscan_map)

11439

In [49]:
with open('data/event_interproscan_map.pkl', "wb") as file:
    pickle.dump(event_interproscan_map, file)

## Merge interproscan results with significant splicing event info.

In [8]:
with open("data/event_interproscan_map.pkl", "rb") as file:
    event_interproscan_map = pickle.load(file)

In [50]:
len(event_interproscan_map)

11439

In [51]:
# merge cell type-specific splicing events with InterProScan results

signif_event_interproscan_map = dict() 
signif_event_interproscan_summary = dict()

for file in os.listdir("data/ctype_exons"):
    if file.endswith("exons.csv"):
        ctype = file.split("_exons.csv")[0]
        print(ctype)
        
        signif_events_df = pd.read_csv(f"data/ctype_exons/{file}", index_col=0)
        
        event_interproscan_dict = {
            ev: event_interproscan_map[ev] for ev in signif_events_df.index 
            if ev in event_interproscan_map
        }

        result = pd.concat(
            [df.assign(event_id=ev, bucket=bucket)
             for ev, buckets in event_interproscan_dict.items()
             for bucket, df in buckets.items()
             if bucket != 'meta' and df is not None],
            ignore_index=True
        )
        # move event and bucket to front
        cols = ['event_id', 'bucket'] + [c for c in result.columns if c not in ('event_id', 'bucket')]
        result = result[cols]
 
        # restrict to splicing events with InterProScan results
        df = result.merge(signif_events_df, left_on="event_id", right_index=True)
        signif_event_interproscan_map[ctype] = df
        
        # summarize interpro results per splicing event
        signif_event_interproscan_summary[ctype] = df.groupby(["event_id", "bucket"]).agg(
            r=('r', lambda x: ' | '.join(map(str, x.unique()))),
            is_specific=('is_specific', lambda x: ' | '.join(map(str, x.unique()))),
            Gene=('Gene', lambda x: ' | '.join(x.unique())),
            interproscan_id=('protein_accession', lambda x: ' | '.join(x.unique())),
            frame_preserving=('frame_preserving', lambda x: ' | '.join(map(str, x.unique()))),
            n_analyses=('analysis', lambda x: len(x.unique())),
            analyses=('analysis', lambda x: ' | '.join(x.unique())),
            signature_descriptions=('signature_description', lambda x: ' | '.join(str(v) for v in x.unique() if pd.notna(v))),
            interpro_descriptions=('interpro_description', lambda x: ' | '.join(x.unique()))
        ).reset_index()

Oligo
VLMC
Endo
Deep_layer_glutamatergic
Astro
OPC
Micro_PVM
All_Neuronal
All_GABAergic
Peri
CGE_Class
Upper_layer_glutamatergic


In [52]:
with open("data/signif_event_interproscan_map.pkl", "wb") as file:
    pickle.dump(signif_event_interproscan_map, file)
    
with open("data/signif_event_interproscan_summary.pkl", "wb") as file:
    pickle.dump(signif_event_interproscan_summary, file)

In [53]:
rec = signif_event_interproscan_summary['Deep_layer_glutamatergic']

rec[(rec['is_specific'] == "True") & (rec['bucket'] == "inclusion")].sort_values('n_analyses', ascending=False).head(10)

,event_id,bucket,r,is_specific,Gene,interproscan_id,frame_preserving,n_analyses,analyses,signature_descriptions,interpro_descriptions
287,ENSG00000056291_ProteinCoding_1,inclusion,0.4471791265064418,True,NPFFR2,ENST00000308744,False,11,CATH-Gene3D | CATH-FunFam | CDD | PIRSR | Pfam...,Rhodopsin 7-helix transmembrane proteins | neu...,"- | G protein-coupled receptor, rhodopsin-like..."
3812,ENSG00000177508_ProteinCoding_1,inclusion,-0.1988403026764951,True,IRX3,ENST00000329734,False,10,CATH-Gene3D | CATH-FunFam | COILS | CDD | Mobi...,Homeodomain-like | Iroquois-class homeobox pro...,- | Homeodomain | KN homeodomain | Iroquois-cl...
2690,ENSG00000146904_ProteinCoding_1,inclusion,0.3414524560565585,True,EPHA1,ENST00000275815,True,9,CATH-Gene3D | CATH-FunFam | PIRSR | Pfam | Pho...,Transferase(Phosphotransferase) domain 1 | Rec...,"- | Serine-threonine/tyrosine-protein kinase, ..."
2421,ENSG00000139880_ProteinCoding_1,inclusion,-0.4879409185382066,True,CDH24,ENST00000397359,True,8,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,Cadherins | Protocadherin beta 4 | Cadherin ta...,- | Cadherin-like | Cadherin-like superfamily
121,ENSG00000010704_ProteinCoding_7,inclusion,-0.1190834969825156,True,HFE,ENST00000353147,True,8,CATH-Gene3D | CATH-FunFam | Pfam | Phobius | S...,Immunoglobulins | Major histocompatibility com...,Immunoglobulin-like fold | - | Immunoglobulin ...
1467,ENSG00000115593_ProteinCoding_1,inclusion,0.2891061506351097,True,SMYD1,ENST00000419482,True,7,CATH-Gene3D | CATH-FunFam | CDD | Pfam | SMART...,SET domain | Histone-lysine N-methyltransferas...,"SET domain superfamily | - | SMYD1, SET domain..."
2857,ENSG00000151914_ProteinCoding_4,inclusion,-0.525019250063025,True,DST,ENST00000680361,True,7,CATH-Gene3D | CATH-FunFam | CDD | Pfam | Phobi...,- | microtubule-actin cross-linking factor 1 |...,- | Spectrin/alpha-actinin | Spectrin repeat
159,ENSG00000017260_ProteinCoding_4,inclusion,0.2051146889237357,True,ATP2C1,ENST00000510168,True,6,CATH-Gene3D | Pfam | Phobius | SMART | SUPERFA...,"Calcium-transporting ATPase, transmembrane dom...","- | Cation-transporting P-type ATPase, N-termi..."
2679,ENSG00000146122_ProteinCoding_1,inclusion,0.2245671444135169,True,DAAM2,ENST00000633794,True,6,CATH-FunFam | Pfam | SMART | SUPERFAMILY | PRO...,Dishevelled associated activator of morphogene...,"- | Formin, FH2 domain | Formin, FH2 domain su..."
3789,ENSG00000176884_ProteinCoding_1,inclusion,0.3713913965532838,True,GRIN1,ENST00000371560,True,6,CATH-FunFam | CDD | PIRSR | Pfam | Phobius | S...,"glutamate receptor ionotropic, NMDA 1 isoform ...","- | Glutamate [NMDA] receptor subunit 1-like, ..."
